In [0]:
from pyspark.sql import functions as F

customer_features_ml = spark.table(
    "workspace.kkbox.customer_features_ml"
)

print("Rows:", customer_features_ml.count())
print("Columns:", len(customer_features_ml.columns))

customer_features_ml.printSchema()

display(
    customer_features_ml.groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

In [0]:
from pyspark.sql import functions as F

model_features = (
    customer_features_ml
    .withColumn(
        "registration_date_parsed",
        F.coalesce(
            F.try_to_date("registration_date", "yyyy-MM-dd"),
            F.try_to_date("registration_date", "yyyyMMdd")
        )
    )
    .withColumn(
        "auto_renew_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("auto_renew_count") / F.col("transaction_count")
        ).otherwise(0.0)
    )
    .withColumn(
        "cancel_rate",
        F.when(
            F.col("transaction_count") > 0,
            F.col("cancel_count") / F.col("transaction_count")
        ).otherwise(0.0)
    )
    .withColumn(
        "days_since_last_transaction",
        F.datediff(
            F.col("latest_expiry_date"),
            F.col("latest_transaction_date")
        )
    )
    .withColumn(
        "registration_year",
        F.year("registration_date_parsed")
    )
    .withColumn(
        "registration_month",
        F.month("registration_date_parsed")
    )
)

display(
    model_features.select(
        "is_churn",
        "age",
        "transaction_count",
        "auto_renew_rate",
        "cancel_rate",
        "days_since_last_transaction",
        "registration_year",
        "registration_month"
    ).limit(20)
)

In [0]:
display(
    model_features.select(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(F.col("registration_date_parsed").isNull(), 1).otherwise(0)
        ).alias("unparsed_registration_dates")
    )
)

In [0]:
display(
    model_features.select(
        F.count("*").alias("total_rows"),
        F.sum(
            F.when(F.col("registration_date_parsed").isNull(), 1).otherwise(0)
        ).alias("missing_registration_dates"),
        F.sum(
            F.when(F.col("days_since_last_transaction").isNull(), 1).otherwise(0)
        ).alias("missing_transaction_gap"),
        F.min("days_since_last_transaction").alias("min_transaction_gap"),
        F.max("days_since_last_transaction").alias("max_transaction_gap"),
        F.min("auto_renew_rate").alias("min_auto_renew_rate"),
        F.max("auto_renew_rate").alias("max_auto_renew_rate"),
        F.min("cancel_rate").alias("min_cancel_rate"),
        F.max("cancel_rate").alias("max_cancel_rate")
    )
)

In [0]:
train_df, test_df = model_features.randomSplit(
    [0.80, 0.20],
    seed=42
)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
from pyspark.sql.window import Window

print("Training distribution")

train_distribution = (
    train_df
    .groupBy("is_churn")
    .count()
    .withColumn(
        "churn_rate",
        F.round(
            F.col("count") /
            F.sum("count").over(Window.partitionBy()) * 100,
            2
        )
    )
    .orderBy("is_churn")
)

display(train_distribution)

In [0]:
train_counts = (
    train_df
    .groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

train_total = train_df.count()

train_distribution = (
    train_counts
    .withColumn(
        "percentage",
        F.round(F.col("count") / F.lit(train_total) * 100, 2)
    )
)

display(train_distribution)

In [0]:
test_counts = (
    test_df
    .groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

test_total = test_df.count()

test_distribution = (
    test_counts
    .withColumn(
        "percentage",
        F.round(F.col("count") / F.lit(test_total) * 100, 2)
    )
)

display(test_distribution)

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

numeric_cols = [
    "age",
    "transaction_count",
    "total_amount_paid",
    "avg_amount_paid",
    "total_plan_price",
    "avg_plan_days",
    "auto_renew_count",
    "cancel_count",
    "auto_renew_rate",
    "cancel_rate",
    "days_since_last_transaction",
    "registration_year",
    "registration_month",
    "has_member_data"
]

categorical_cols = [
    "city",
    "gender",
    "registered_via"
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{col}_index",
        outputCol=f"{col}_encoded"
    )
    for col in categorical_cols
]

assembler = VectorAssembler(
    inputCols=(
        numeric_cols +
        [f"{col}_encoded" for col in categorical_cols]
    ),
    outputCol="features",
    handleInvalid="keep"
)

preprocessing_pipeline = Pipeline(
    stages=indexers + encoders + [assembler]
)

preprocessing_model = preprocessing_pipeline.fit(train_df)

train_prepared = preprocessing_model.transform(train_df)
test_prepared = preprocessing_model.transform(test_df)

display(
    train_prepared.select(
        "is_churn",
        "features"
    ).limit(5)
)

In [0]:
train_non_churn = 149954
train_churn = 10155
train_total = train_non_churn + train_churn

non_churn_weight = train_total / (2 * train_non_churn)
churn_weight = train_total / (2 * train_churn)

print("Non-churn weight:", non_churn_weight)
print("Churn weight:", churn_weight)

In [0]:
train_weighted = (
    train_prepared
    .withColumn(
        "class_weight",
        F.when(
            F.col("is_churn") == 1,
            F.lit(churn_weight)
        ).otherwise(
            F.lit(non_churn_weight)
        )
    )
)

display(
    train_weighted
    .groupBy("is_churn")
    .agg(
        F.count("*").alias("customers"),
        F.first("class_weight").alias("class_weight")
    )
    .orderBy("is_churn")
)

In [0]:
model_features = (
    model_features
    .withColumn(
        "registration_year",
        F.coalesce(F.col("registration_year"), F.lit(-1))
    )
    .withColumn(
        "registration_month",
        F.coalesce(F.col("registration_month"), F.lit(-1))
    )
)

display(
    model_features.select(
        "registration_date",
        "registration_year",
        "registration_month"
    ).limit(20)
)

In [0]:
display(
    model_features.select(
        F.sum(
            F.when(F.isnan("registration_year"), 1).otherwise(0)
        ).alias("nan_registration_year"),
        F.sum(
            F.when(F.isnan("registration_month"), 1).otherwise(0)
        ).alias("nan_registration_month"),
        F.sum(
            F.when(F.col("registration_year").isNull(), 1).otherwise(0)
        ).alias("null_registration_year"),
        F.sum(
            F.when(F.col("registration_month").isNull(), 1).otherwise(0)
        ).alias("null_registration_month")
    )
)


In [0]:
train_df, test_df = model_features.randomSplit(
    [0.80, 0.20],
    seed=42
)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_prepared = preprocessing_model.transform(train_df)
test_prepared = preprocessing_model.transform(test_df)

print("Preprocessing completed successfully.")

In [0]:
display(
    train_prepared.select(
        "is_churn",
        "features"
    ).limit(5)
)

In [0]:
train_class_counts = (
    train_df
    .groupBy("is_churn")
    .count()
    .collect()
)

class_counts = {
    int(row["is_churn"]): row["count"]
    for row in train_class_counts
}

train_total = sum(class_counts.values())

non_churn_weight = train_total / (2 * class_counts[0])
churn_weight = train_total / (2 * class_counts[1])

print("Non-churn weight:", non_churn_weight)
print("Churn weight:", churn_weight)

In [0]:
train_weighted = (
    train_prepared
    .withColumn(
        "class_weight",
        F.when(
            F.col("is_churn") == 1,
            F.lit(churn_weight)
        ).otherwise(
            F.lit(non_churn_weight)
        )
    )
)

display(
    train_weighted
    .groupBy("is_churn")
    .agg(
        F.count("*").alias("customers"),
        F.first("class_weight").alias("class_weight")
    )
    .orderBy("is_churn")
)

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_churn",
    weightCol="class_weight",
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_weighted)

print("Model trained successfully.")

In [0]:
test_predictions = lr_model.transform(test_prepared)

display(
    test_predictions.select(
        "is_churn",
        "probability",
        "prediction"
    ).limit(20)
)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

roc_auc = roc_evaluator.evaluate(test_predictions)
pr_auc = pr_evaluator.evaluate(test_predictions)

print("ROC-AUC:", round(roc_auc, 4))
print("PR-AUC:", round(pr_auc, 4))

In [0]:
confusion_matrix = (
    test_predictions
    .groupBy("is_churn", "prediction")
    .count()
    .orderBy("is_churn", "prediction")
)

display(confusion_matrix)

In [0]:
cm = {
    (int(row["is_churn"]), int(row["prediction"])): row["count"]
    for row in confusion_matrix.collect()
}

true_negative = cm.get((0, 0), 0)
false_positive = cm.get((0, 1), 0)
false_negative = cm.get((1, 0), 0)
true_positive = cm.get((1, 1), 0)

precision = (
    true_positive /
    (true_positive + false_positive)
    if (true_positive + false_positive) > 0
    else 0
)

recall = (
    true_positive /
    (true_positive + false_negative)
    if (true_positive + false_negative) > 0
    else 0
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1:", round(f1, 4))

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="is_churn",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_weighted)

print("Random Forest trained successfully.")

In [0]:
rf_predictions = rf_model.transform(test_prepared)

display(
    rf_predictions.select(
        "is_churn",
        "probability",
        "prediction"
    ).limit(20)
)

In [0]:
rf_roc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

rf_pr_evaluator = BinaryClassificationEvaluator(
    labelCol="is_churn",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

rf_roc_auc = rf_roc_evaluator.evaluate(rf_predictions)
rf_pr_auc = rf_pr_evaluator.evaluate(rf_predictions)

print("Random Forest ROC-AUC:", round(rf_roc_auc, 4))
print("Random Forest PR-AUC:", round(rf_pr_auc, 4))

In [0]:
rf_confusion_matrix = (
    rf_predictions
    .groupBy("is_churn", "prediction")
    .count()
    .orderBy("is_churn", "prediction")
)

display(rf_confusion_matrix)

In [0]:
rf_cm = {
    (int(row["is_churn"]), int(row["prediction"])): row["count"]
    for row in rf_confusion_matrix.collect()
}

rf_true_negative = rf_cm.get((0, 0), 0)
rf_false_positive = rf_cm.get((0, 1), 0)
rf_false_negative = rf_cm.get((1, 0), 0)
rf_true_positive = rf_cm.get((1, 1), 0)

rf_precision = (
    rf_true_positive /
    (rf_true_positive + rf_false_positive)
    if (rf_true_positive + rf_false_positive) > 0
    else 0
)

rf_recall = (
    rf_true_positive /
    (rf_true_positive + rf_false_negative)
    if (rf_true_positive + rf_false_negative) > 0
    else 0
)

rf_f1 = (
    2 * rf_precision * rf_recall /
    (rf_precision + rf_recall)
    if (rf_precision + rf_recall) > 0
    else 0
)

print("Random Forest Precision:", round(rf_precision, 4))
print("Random Forest Recall:", round(rf_recall, 4))
print("Random Forest F1:", round(rf_f1, 4))

In [0]:
feature_importance = (
    rf_model
    .featureImportances
)

importance_df = spark.createDataFrame(
    [
        (i, float(feature_importance[i]))
        for i in range(len(feature_importance))
    ],
    ["feature_index", "importance"]
)

display(
    importance_df
    .orderBy(F.desc("importance"))
    .limit(20)
)

In [0]:
import json

feature_metadata = train_prepared.schema["features"].metadata

ml_attributes = feature_metadata["ml_attr"]

feature_names = {}

for attr_type in ["numeric", "binary"]:
    for attr in ml_attributes.get("attrs", {}).get(attr_type, []):
        feature_names[attr["idx"]] = attr["name"]

feature_importance_named = [
    (
        idx,
        feature_names.get(idx, f"feature_{idx}"),
        float(feature_importance[idx])
    )
    for idx in range(len(feature_importance))
]

feature_importance_df = spark.createDataFrame(
    feature_importance_named,
    ["feature_index", "feature_name", "importance"]
)

display(
    feature_importance_df
    .orderBy(F.desc("importance"))
    .limit(20)
)

In [0]:
model_path = "/Volumes/workspace/kkbox/models/churn_random_forest"

rf_model.write().overwrite().save(model_path)

print("Random Forest model saved to:")
print(model_path)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.kkbox.models;

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.kkbox.models
""")

print("Models volume created or already exists.")

In [0]:
model_path = "/Volumes/workspace/kkbox/models/churn_random_forest"

rf_model.write().overwrite().save(model_path)

print("Random Forest model saved successfully.")
print(model_path)

In [0]:
from pyspark.ml.classification import RandomForestClassificationModel

rf_model_check = RandomForestClassificationModel.load(model_path)

print("Model loaded successfully.")
print("Number of trees:", rf_model_check.getNumTrees)

In [0]:
feature_importance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.kkbox.churn_feature_importance")

print("Feature importance table saved successfully.")

In [0]:
display(
    spark.table("workspace.kkbox.churn_feature_importance")
    .orderBy(F.desc("importance"))
    .limit(20)
)